# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [6]:
# Write your code below.
from dotenv import load_dotenv
load_dotenv()


True

In [7]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [8]:
import os
from glob import glob
import dask.dataframe as dd
import pandas as pd

# Load environment variable
price_data_path = os.getenv('PRICE_DATA')
print(f"Price data path: {price_data_path}")

# Use glob to find all parquet files (recursive search needed)
parquet_files = glob(os.path.join(price_data_path, "**", "*.parquet"), recursive=True)
print(f"Found {len(parquet_files)} parquet files")
print("Sample files:", parquet_files[:3])

Price data path: ../../05_src/data/prices/
Found 2823 parquet files
Sample files: ['../../05_src/data/prices\\ACC\\ACC_2004\\part.0.parquet', '../../05_src/data/prices\\ACC\\ACC_2004\\part.1.parquet', '../../05_src/data/prices\\ACC\\ACC_2005\\part.0.parquet']


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [9]:
dd_price = dd.read_parquet(
    os.path.join(price_data_path, "**", "*.parquet"),
    engine='pyarrow'
)

# Verify the data loaded correctly
sample_data = dd_price.head()

# Sort by ticker and date for proper time series operations
dd_price = dd_price.sort_values(['ticker', 'Date'])

def add_features(partition):
    """Function to add features to each partition"""
    # Sort partition by ticker and date
    partition = partition.sort_values(['ticker', 'Date'])
    
    # Create lag features using groupby within partition
    partition['Close_lag_1'] = partition.groupby('ticker')['Close'].shift(1)
    partition['Adj_Close_lag_1'] = partition.groupby('ticker')['Adj Close'].shift(1)
    
    # Add returns: (Close / Close_lag_1) - 1
    partition['returns'] = (partition['Close'] / partition['Close_lag_1']) - 1
    
    # Add hi_lo_range: High - Low
    partition['hi_lo_range'] = partition['High'] - partition['Low']
    
    return partition

# Apply the function to each partition
print("Applying feature engineering to each partition...")
dd_feat = dd_price.map_partitions(
    add_features,
    meta=dd_price._meta.assign(
        Close_lag_1='f8',
        Adj_Close_lag_1='f8', 
        returns='f8',
        hi_lo_range='f8'
    )
)

print(f"New columns: {dd_feat.columns.tolist()}")

# Verify features were created by checking a sample
print("\nVerifying features with sample data:")
sample_with_features = dd_feat.head(10)
feature_cols = ['Close_lag_1', 'Adj_Close_lag_1', 'returns', 'hi_lo_range']
print(sample_with_features[['ticker', 'Date', 'Close'] + feature_cols])

Applying feature engineering to each partition...
New columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'ticker', 'Year', 'Close_lag_1', 'Adj_Close_lag_1', 'returns', 'hi_lo_range']

Verifying features with sample data:
  ticker       Date      Close  Close_lag_1  Adj_Close_lag_1   returns  \
0    ACC 2004-08-16  17.500000          NaN              NaN       NaN   
1    ACC 2004-08-17  17.340000    17.500000         8.894733 -0.009143   
2    ACC 2004-08-18  17.110001    17.340000         8.813408 -0.013264   
3    ACC 2004-08-19  17.090000    17.110001         8.696510 -0.001169   
4    ACC 2004-08-20  17.400000    17.090000         8.686344  0.018139   
5    ACC 2004-08-23  17.420000    17.400000         8.843906  0.001149   
6    ACC 2004-08-24  17.620001    17.420000         8.854072  0.011481   
7    ACC 2004-08-25  17.549999    17.620001         8.955727 -0.003973   
8    ACC 2004-08-26  17.510000    17.549999         8.920147 -0.002279   
9    AC

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [10]:
# Write your code below.

df_feat = dd_feat.compute()

# Sort by ticker and date to ensure proper rolling window calculation
df_feat = df_feat.sort_values(['ticker', 'Date']).reset_index(drop=True)

print(f"\n=== ADDING MOVING AVERAGE WITH PANDAS ===")
print("Adding 10-day moving average of returns...")

# Add 10-day moving average of returns
df_feat['returns_ma_10'] = df_feat.groupby('ticker')['returns'].rolling(
    window=10, min_periods=1
).mean().reset_index(0, drop=True)

# Display results
all_feature_cols = ['Close_lag_1', 'Adj_Close_lag_1', 'returns', 'hi_lo_range', 'returns_ma_10']

print("New features summary statistics:")
print(df_feat[all_feature_cols].describe().round(6))

print(f"\nSample data for first ticker:")
first_ticker = df_feat['ticker'].iloc[0]
sample_data = df_feat[df_feat['ticker'] == first_ticker].head(15)
display_cols = ['Date', 'ticker', 'Close', 'Close_lag_1', 'returns', 'hi_lo_range', 'returns_ma_10']
print(sample_data[display_cols].round(6))

print(f"\n=== FINAL DATASET INFO ===")
print(f"Total rows: {len(df_feat):,}")
print(f"Number of tickers: {df_feat['ticker'].nunique()}")
print(f"Date range: {df_feat['Date'].min()} to {df_feat['Date'].max()}")
print(f"Features created with Dask: 4")
print(f"Features added with pandas: 1 (moving average)")

# Data quality check
print(f"\nData quality - Missing values in new features:")
for col in all_feature_cols:
    missing = df_feat[col].isnull().sum()
    pct_missing = (missing / len(df_feat)) * 100
    print(f"  {col}: {missing:,} ({pct_missing:.2f}%)")




=== ADDING MOVING AVERAGE WITH PANDAS ===
Adding 10-day moving average of returns...
New features summary statistics:
         Close_lag_1  Adj_Close_lag_1        returns    hi_lo_range  \
count  328208.000000    328208.000000  328204.000000  328297.000000   
mean       30.203553        24.680076       0.001854       0.787118   
std        34.020577        32.989493       0.620336       1.621085   
min         0.150000         0.000846      -0.751196       0.000000   
25%         9.550000         5.623908      -0.009404       0.160000   
50%        19.500000        13.670000       0.000000       0.400000   
75%        37.580002        28.542073       0.009569       0.889999   
max       604.250000       604.250000     332.333320     117.500000   

       returns_ma_10  
count  328212.000000  
mean        0.001944  
std         0.199076  
min        -0.144276  
25%        -0.002918  
50%         0.000305  
75%         0.003680  
max        33.277145  

Sample data for first ticker:
   

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
No, it was not strictly necessary. Dask does support rolling window operations, including

+ Would it have been better to do it in Dask? Why?
Yes, it would have been better to do it in Dask for several reasons:
Memory efficiency: Dask processes data in chunks, so it can handle datasets larger than RAM without memory issues.
Parallel processing: Dask can compute the moving average for different tickers in parallel, making it faster for large datasets.
Lazy evaluation: Dask delays computation until needed, allowing for optimization of the entire computation graph.
Consistency: Keeping everything in Dask maintains a consistent workflow and avoids the overhead of converting between formats.
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.